# 5. Tools、Resources与Prompts

前四节的 MCP 服务只提供工具（Tools）。MCP 服务还能提供另外两类能力：**资料（Resources）**和**提示词模板（Prompts）**。客户端发现它们之后，它们会不会像工具一样自动交给 Agent？

本节使用第十章客服知识库中的套餐说明，在同一个服务里注册三类能力，再用客户端分别发现和获取，检查 `MCPAdapter` 的封装范围，最后把资料和模板实际交给 Agent。

学习目标：

1. 说出三类能力分别由谁决定使用。
2. 用 `@mcp.tool`、`@mcp.resource`、`@mcp.prompt` 注册三类能力。
3. 用客户端发现并获取三类能力，比较它们的描述和返回内容。
4. 验证 `MCPAdapter` 只把 Tools 转成 LangChain 工具。
5. 观察 Resource、Prompt 进入 Agent 的消息，以及模型按需读取资源的真实调用。

5.1–5.7 的发现与获取实验只在本地运行，不需要模型密钥。新增的 **5.8 会请求真实 DeepSeek 模型**，需在项目 `conf/.env` 配置 DEEPSEEK_API_KEY，复用第一、二节的模型配置。客户端使用进程内连接；服务脚本供第六节复用。选择项目 `.venv` 内核后，按顺序执行。

## 5.1 三类能力由谁决定使用

![MCP 三类能力由谁决定使用，以及怎样进入模型](assets/17-mcp-three-primitives.svg)

MCP 规范按**由谁控制**来区分服务端的三类能力：

| 能力 | 由谁决定使用 | 规范中的例子 | 本节的套餐示例 |
|---|---|---|---|
| Tools | 模型 | 发起 API 写入请求、写文件 | `get_plan`：模型判断需要时，查询某个套餐 |
| Resources | 应用 | 文件内容、git 历史 | `plans://catalog`：应用决定是否把套餐对比表放进上下文 |
| Prompts | 用户 | 斜杠命令、菜单选项 | `plan_answer`：用户选择“套餐答疑”模板，再填写问题 |

Agent 的工具列表只放模型**可以自主调用**的能力，所以只有 Tools 会进入。Resources 和 Prompts 由应用或用户决定何时使用，需要应用代码把它们放进消息。后面的实验逐一验证这一点。

上表是三类能力通常的交互方式。应用可以另外封装一个资料读取工具，让模型按需请求 Resource；5.8 会观察这个应用层扩展。

## 5.2 环境准备

继续使用 FastMCP 的服务端与客户端，以及第一节的 `MCPAdapter`。路径定位同时支持从项目根目录和章节目录运行。

In [1]:
import json
import runpy
import sys
from importlib.metadata import version
from pathlib import Path
from typing import Literal

from fastmcp import Client, FastMCP
from langchain.mcp import MCPAdapter

PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("请从 lang-chain-learn 项目或其子目录运行本笔记。")

SCRIPT_FILE = PROJECT_ROOT / "Chapter11_MCP" / "servers" / "plan_server.py"

print("Python:", sys.executable)
for package in ["fastmcp", "mcp", "langchain"]:
    print(f"{package}: {version(package)}")
print("套餐服务脚本存在:", SCRIPT_FILE.is_file())

Python: /Users/skk/Developer/lang-chain-learn/.venv/bin/python
fastmcp: 4.1.0
mcp: 2.3.0
langchain: 1.4.3
套餐服务脚本存在: True


/var/folders/14/yh05p4nj2z70mjs568rzs9gh0000gn/T/ipykernel_58717/561118346.py:9: LangChainBetaWarning: `langchain.mcp` is in beta. It is actively being worked on, so the API may change.
  from langchain.mcp import MCPAdapter


## 5.3 在同一个服务中注册三类能力

### 5.3.1 套餐数据与查询工具

套餐数据摘自 [第十章知识库](../Chapter10_RAG/knowledge.txt) 的“二、套餐说明”，每档只保留四个字段，仅用于教学。

工具注册与第三节相同。`Literal[...]` 把参数限定为四个套餐名称，客户端看到的 Schema 中会出现 `enum`。

`call_log` 记录函数何时被执行，作用与第三节的 `function_entries` 相同；独立脚本中改为 stderr 日志。

In [2]:
# 创建服务端对象,此时未监听任何端口或通道，也没有进入运行状态、未被调用 run()
# 创建服务端对象: plan_server = FastMCP(name="PlanService") 只是内存里的对象，注册表是空的
# 注册内容: @plan_server.tool / @plan_server.resource / @plan_server.prompt 往注册表里登记工具/资源/提示词
# 启动服务: plan_server.run(transport=...) 这时才真正开始运行，监听并响应客户端请求
plan_server = FastMCP(name="PlanService")
call_log = []

PLANS = {
    "试用版": {
        "价格": "0 元，注册后 14 天",
        "成员人数上限": "1 人",
        "月度AI问答额度": "200 次",
        "API调用额度": "不支持",
    },
    "基础版": {
        "价格": "99 元 / 用户 / 月",
        "成员人数上限": "10 人",
        "月度AI问答额度": "5000 次",
        "API调用额度": "每月 10000 次",
    },
    "专业版": {
        "价格": "199 元 / 用户 / 月",
        "成员人数上限": "50 人",
        "月度AI问答额度": "30000 次",
        "API调用额度": "每月 80000 次",
    },
    "企业版": {
        "价格": "按年签约，不公开标价",
        "成员人数上限": "按合同约定",
        "月度AI问答额度": "按合同约定",
        "API调用额度": "按合同约定",
    },
}


@plan_server.tool
def get_plan(plan: Literal["试用版", "基础版", "专业版", "企业版"]) -> dict:
    """查询一个套餐的价格、成员人数上限、月度 AI 问答额度和 API 调用额度。"""
    call_log.append(("tool", "get_plan", plan))
    return {"套餐": plan, **PLANS[plan]}

### 5.3.2 Resource：用 URI 标识的一份资料

`@mcp.resource` 必须先传入 URI，常用参数如下（签名省略了本节不用的参数）：

```python
@mcp.resource(uri, *, name=None, description=None, mime_type=None, ...)
```

| 参数 | 含义及本例用法 |
|---|---|
| `uri` | 资料的唯一标识，必填。`plans://catalog` 中的 `plans` 是自定义前缀，它不是文件路径，也不是网址 |
| `name` | 资料名称；省略时使用函数名 `plan_catalog` |
| `description` | 资料说明；省略时使用 docstring |
| `mime_type` | 内容格式；本例返回 Markdown 表格，传 `"text/markdown"` |

被装饰的函数在**客户端读取资料时**才执行，返回的字符串就是资料内容。

In [3]:
@plan_server.resource("plans://catalog", mime_type="text/markdown")
def plan_catalog() -> str:
    """四档标准订阅套餐的对比表。"""
    call_log.append(("resource", "plans://catalog"))
    fields = list(PLANS["试用版"])
    lines = [
        "| 套餐 | " + " | ".join(fields) + " |",
        "|---" * (len(fields) + 1) + "|",
    ]
    for name, plan in PLANS.items():
        lines.append(f"| {name} | " + " | ".join(plan[field] for field in fields) + " |")
    return "\n".join(lines)

### 5.3.3 Prompt：可以填写参数的消息模板

`@mcp.prompt` 与 `@mcp.tool` 一样，默认使用函数名作为名称、docstring 作为描述。函数参数就是模板参数：`question` 必填，`style` 有默认值，可以省略。

函数返回字符串时，FastMCP 把它包装成一条 `role="user"` 的消息。模板只说明“怎样回答”，并要求依据套餐资料；资料本身仍需要应用另外提供。

In [4]:
@plan_server.prompt
def plan_answer(question: str, style: str = "简洁") -> str:
    """按客服口径回答套餐问题的提示词模板。"""
    call_log.append(("prompt", "plan_answer", style))
    return (
        "你是 atguigu助手的客服。请只依据套餐资料回答用户的问题；"
        "资料没有写明的内容，说明需要人工确认。\n"
        f"回答风格：{style}。\n"
        f"用户问题：{question}"
    )# 返回HumanMessage

先在**服务端**确认三类能力都已登记。三个方法分别返回服务端的工具、资料和模板对象。

In [5]:
registered_tools = await plan_server.list_tools()
registered_resources = await plan_server.list_resources()
registered_prompts = await plan_server.list_prompts()

print("工具:", [(type(item).__name__, item.name) for item in registered_tools])
print("资料:", [(type(item).__name__, str(item.uri)) for item in registered_resources])
print("模板:", [(type(item).__name__, item.name) for item in registered_prompts])
print("注册后，函数执行记录:", call_log)

工具: [('FunctionTool', 'get_plan')]
资料: [('FunctionResource', 'plans://catalog')]
模板: [('FunctionPrompt', 'plan_answer')]
注册后，函数执行记录: []


三个函数都已登记，但还没有执行。

## 5.4 客户端分别发现三类能力

三类能力各有一个“列出”接口。它们只返回描述，不执行函数：

| 接口 | 等待后的返回值 | 描述中的关键字段 |
|---|---|---|
| `await client.list_tools()` | `list[Tool]` | name、description、input_schema |
| `await client.list_resources()` | `list[Resource]` | uri、name、description、mime_type |
| `await client.list_prompts()` | `list[Prompt]` | name、description、arguments |

三个接口都有 `max_pages=250` 参数，限制自动分页的次数；本节使用默认值。带参数的 URI（例如 `plans://{plan}`）称为资源模板，要用 `list_resource_templates()` 另外发现，本节放在小练习中。

In [6]:
from rich import print as rprint
client = Client(plan_server)

async with client:
    tool_specs = await client.list_tools()
    resource_specs = await client.list_resources()
    prompt_specs = await client.list_prompts()

for label, specs in [("工具", tool_specs), ("资料", resource_specs), ("模板", prompt_specs)]:
    print(f"{label}:", type(specs).__name__, [type(spec).__name__ for spec in specs])
    print(type(specs[0]))
print()

rprint("完整Tool:\n",tool_specs[0])
print("Tool 的 input_schema:")
print(json.dumps(tool_specs[0].input_schema, ensure_ascii=False, indent=2))
print()

catalog_spec = resource_specs[0]
rprint("完整Resource:\n", catalog_spec.model_dump(exclude_none=True))
print("Resource 的描述:")
print("  uri:", str(catalog_spec.uri))
print("  name:", catalog_spec.name)
print("  description:", catalog_spec.description)
print("  mime_type:", catalog_spec.mime_type)
print()

print("Prompt 的 arguments:")
rprint("完整Prompt :\n", prompt_specs[0])
for argument in prompt_specs[0].arguments:
    print(" ", argument.model_dump(exclude_none=True))
print()
print("发现后，函数执行记录:", call_log)

工具: list ['Tool']
<class 'mcp_types._types.Tool'>
资料: list ['Resource']
<class 'mcp_types._types.Resource'>
模板: list ['Prompt']
<class 'mcp_types._types.Prompt'>



完整Tool:

Tool(
    name='get_plan',
    title='Get Plan',
    description='查询一个套餐的价格、成员人数上限、月度 AI 问答额度和 API 调用额度。',
    input_schema={
        'type': 'object',
        'additionalProperties': False,
        'properties': {'plan': {'enum': ['试用版', '基础版', '专业版', '企业版'], 'type': 'string'}},
        'required': ['plan']
    },
    execution=None,
    output_schema={'additionalProperties': True, 'type': 'object'},
    icons=None,
    annotations=None,
    meta={'fastmcp': {'tags': []}}
)

Tool 的 input_schema:
{
  "type": "object",
  "additionalProperties": false,
  "properties": {
    "plan": {
      "enum": [
        "试用版",
        "基础版",
        "专业版",
        "企业版"
      ],
      "type": "string"
    }
  },
  "required": [
    "plan"
  ]
}



完整Resource:

{
    'name': 'plan_catalog',
    'uri': 'plans://catalog',
    'description': '四档标准订阅套餐的对比表。',
    'mime_type': 'text/markdown',
    'meta': {'fastmcp': {'tags': []}}
}

Resource 的描述:
  uri: plans://catalog
  name: plan_catalog
  description: 四档标准订阅套餐的对比表。
  mime_type: text/markdown

Prompt 的 arguments:


完整Prompt :

Prompt(
    name='plan_answer',
    title=None,
    description='按客服口径回答套餐问题的提示词模板。',
    arguments=[
        PromptArgument(name='question', title=None, description=None, required=True),
        PromptArgument(name='style', title=None, description=None, required=False)
    ],
    icons=None,
    meta={'fastmcp': {'tags': []}}
)

  {'name': 'question', 'required': True}
  {'name': 'style', 'required': False}

发现后，函数执行记录: []


三份描述对参数的说明方式不同：

| 能力 | 参数如何描述 | 由谁填写 |
|---|---|---|
| Tool | `input_schema` 是完整的 JSON Schema，包含类型、`enum` 和 `required` | 模型按 Schema 生成参数 |
| Resource | 没有参数，用 URI 定位资料 | 应用选择要读取的 URI |
| Prompt | `arguments` 只有名称和是否必填 | 用户或界面填写；MCP 协议中模板参数都按字符串传递 |

发现之后，`call_log` 仍为空：拿到描述不等于执行了函数，也不等于拿到了资料内容。

## 5.5 分别获取三类能力

### 5.5.1 读取Resource

Client读取后返回内容块

```python
await client.read_resource(uri, *, version=None, meta=None)
```

| 参数 / 返回值 | 含义 |
|---|---|
| `uri` | 要读取的资料 URI，传字符串即可 |
| `version`、`meta` | 版本选择与请求元数据，本节不用 |
| 等待后的返回值 | 内容列表。一份资料可以包含多段内容：文本是 `TextResourceContents`，二进制是 `BlobResourceContents` |

In [7]:
async with client:
    catalog_contents = await client.read_resource("plans://catalog")

rprint("完整read_resource返回值:\n",catalog_contents)
first_content = catalog_contents[0]
print("返回容器:", type(catalog_contents).__name__, "长度:", len(catalog_contents))
print("内容类型:", type(first_content).__name__)
print("uri:", first_content.uri, "mime_type:", first_content.mime_type)
print("资料文本:")
print(first_content.text)
print()
print("读取后，函数执行记录:", call_log)

完整read_resource返回值:

[
    TextResourceContents(
        uri='plans://catalog',
        mime_type='text/markdown',
        meta=None,
        text='| 套餐 | 价格 | 成员人数上限 | 月度AI问答额度 | API调用额度 |\n|---|---|---|---|---|\n| 试用版 | 0 
元，注册后 14 天 | 1 人 | 200 次 | 不支持 |\n| 基础版 | 99 元 / 用户 / 月 | 10 人 | 5000 次 | 每月 10000 次 |\n| 
专业版 | 199 元 / 用户 / 月 | 50 人 | 30000 次 | 每月 80000 次 |\n| 企业版 | 按年签约，不公开标价 | 按合同约定 | 
按合同约定 | 按合同约定 |'
    )
]

返回容器: list 长度: 1
内容类型: TextResourceContents
uri: plans://catalog mime_type: text/markdown
资料文本:
| 套餐 | 价格 | 成员人数上限 | 月度AI问答额度 | API调用额度 |
|---|---|---|---|---|
| 试用版 | 0 元，注册后 14 天 | 1 人 | 200 次 | 不支持 |
| 基础版 | 99 元 / 用户 / 月 | 10 人 | 5000 次 | 每月 10000 次 |
| 专业版 | 199 元 / 用户 / 月 | 50 人 | 30000 次 | 每月 80000 次 |
| 企业版 | 按年签约，不公开标价 | 按合同约定 | 按合同约定 | 按合同约定 |

读取后，函数执行记录: [('resource', 'plans://catalog')]


### 5.5.2 获取Prompt，修改一个条件

```python
await client.get_prompt(name, arguments=None, *, version=None, meta=None)
```

| 参数 / 返回值 | 含义 |
|---|---|
| `name` | 模板名称，例如 `"plan_answer"` |
| `arguments` | 模板参数字典；本例 `question` 必填，`style` 可以省略 |
| `version`、`meta` | 版本选择与请求元数据，本节不用 |
| 等待后的返回值 | `GetPromptResult`：`description` 是模板描述，`messages` 是填好参数的消息列表 |

同一个问题获取两次：第一次省略 `style`，第二次改为 `"详细"`。先预测两次消息文本有什么不同。

In [8]:
question = "专业版最多能加多少人？"

async with client:
    prompt_default = await client.get_prompt("plan_answer", {"question": question})
    prompt_detailed = await client.get_prompt(
        "plan_answer", {"question": question, "style": "详细"}
    )

first_message = prompt_default.messages[0]
rprint("prompt_default:\n",prompt_default)
rprint("prompt_detailed:\n",prompt_detailed)
print("返回类型:", type(prompt_default).__name__)
print("description:", prompt_default.description)
print("消息条数:", len(prompt_default.messages))
print("消息类型:", type(first_message).__name__, "role:", first_message.role)
print("内容类型:", type(first_message.content).__name__)
print()
print("省略 style 的消息:")
print(first_message.content.text)
print()
print("style=详细 的消息:")
print(prompt_detailed.messages[0].content.text)
print()
print("最新两条函数执行记录:", call_log[-2:])

prompt_default:

GetPromptResult(
    meta={'io.modelcontextprotocol/serverInfo': {'name': 'PlanService', 'version': '4.1.0'}},
    description='按客服口径回答套餐问题的提示词模板。',
    messages=[
        PromptMessage(
            role='user',
            content=TextContent(
                type='text',
                text='你是 
atguigu助手的客服。请只依据套餐资料回答用户的问题；资料没有写明的内容，说明需要人工确认。\n回答风格：简洁。\n用户问
题：专业版最多能加多少人？',
                annotations=None,
                meta=None
            )
        )
    ],
    result_type='complete'
)

prompt_detailed:

GetPromptResult(
    meta={'io.modelcontextprotocol/serverInfo': {'name': 'PlanService', 'version': '4.1.0'}},
    description='按客服口径回答套餐问题的提示词模板。',
    messages=[
        PromptMessage(
            role='user',
            content=TextContent(
                type='text',
                text='你是 
atguigu助手的客服。请只依据套餐资料回答用户的问题；资料没有写明的内容，说明需要人工确认。\n回答风格：详细。\n用户问
题：专业版最多能加多少人？',
                annotations=None,
                meta=None
            )
        )
    ],
    result_type='complete'
)

返回类型: GetPromptResult
description: 按客服口径回答套餐问题的提示词模板。
消息条数: 1
消息类型: PromptMessage role: user
内容类型: TextContent

省略 style 的消息:
你是 atguigu助手的客服。请只依据套餐资料回答用户的问题；资料没有写明的内容，说明需要人工确认。
回答风格：简洁。
用户问题：专业版最多能加多少人？

style=详细 的消息:
你是 atguigu助手的客服。请只依据套餐资料回答用户的问题；资料没有写明的内容，说明需要人工确认。
回答风格：详细。
用户问题：专业版最多能加多少人？

最新两条函数执行记录: [('prompt', 'plan_answer', '简洁'), ('prompt', 'plan_answer', '详细')]


省略 `style` 时，服务端使用默认值“简洁”。两次获取都只返回消息，模板中的“依据套餐资料回答”不会自动附上对比表。

### 5.5.3 调用Tool

调用方式与第三节相同：`call_tool` 执行函数，返回 `CallToolResult`。

In [9]:
async with client:
    plan_result = await client.call_tool("get_plan", {"plan": "专业版"})

print("返回类型:", type(plan_result).__name__, "is_error:", plan_result.is_error)
print("data:", plan_result.data)
print("最新函数执行记录:", call_log[-1])
rprint(plan_result)

返回类型: CallToolResult is_error: False
data: {'套餐': '专业版', '价格': '199 元 / 用户 / 月', '成员人数上限': '50 人', '月度AI问答额度': '30000 次', 'API调用额度': '每月 80000 次'}
最新函数执行记录: ('tool', 'get_plan', '专业版')


CallToolResult(
    content=[
        TextContent(
            type='text',
            text='{"套餐":"专业版","价格":"199 元 / 用户 / 月","成员人数上限":"50 人","月度AI问答额度":"30000 
次","API调用额度":"每月 80000 次"}',
            annotations=None,
            meta=None
        )
    ],
    structured_content={
        '套餐': '专业版',
        '价格': '199 元 / 用户 / 月',
        '成员人数上限': '50 人',
        '月度AI问答额度': '30000 次',
        'API调用额度': '每月 80000 次'
    },
    meta={'io.modelcontextprotocol/serverInfo': {'name': 'PlanService', 'version': '4.1.0'}},
    data={
        '套餐': '专业版',
        '价格': '199 元 / 用户 / 月',
        '成员人数上限': '50 人',
        '月度AI问答额度': '30000 次',
        'API调用额度': '每月 80000 次'
    },
    is_error=False
)

三种获取方式对照如下：

| 能力 | 获取接口 | 等待后的返回值 | 得到的内容 |
|---|---|---|---|
| Tool | `call_tool(name, arguments)` | `CallToolResult` | 函数执行结果 |
| Resource | `read_resource(uri)` | `TextResourceContents` 等组成的列表 | 资料内容 |
| Prompt | `get_prompt(name, arguments)` | `GetPromptResult` | 填好参数的消息 |

三次获取都执行了服务端函数，但结果只回到客户端。这些内容是否交给模型，由调用它们的代码决定。

## 5.6 只有Tools进入Agent的工具列表

`MCPAdapter.list_tools()` 返回的列表，就是交给 `create_agent(tools=...)` 的工具。检查它包含哪些能力，以及 Adapter 有没有读取资料或模板的方法。

In [10]:
# 将服务器适配langchain
async with MCPAdapter(plan_server) as adapter:
    tools_for_agent = await adapter.list_tools()

print("适配后的工具:", [(type(item).__name__, item.name) for item in tools_for_agent])
print("MCPAdapter 的公开成员:", [name for name in dir(adapter) if not name.startswith("_")])
#print(await adapter.client.get_prompt(""))

适配后的工具: [('StructuredTool', 'get_plan')]
MCPAdapter 的公开成员: ['client', 'list_tools']


适配后只有 `get_plan`。`MCPAdapter` 的公开成员只有 `list_tools` 和 `client`：`client` 是它底层的 FastMCP Client（见第三节 3.7.1），读取资料和模板仍要通过 Client 的 `read_resource()`、`get_prompt()`，也就是 5.5 的做法。

所以，**发现 Resource 或 Prompt 不等于它们已经进入模型上下文**。如果要用套餐对比表或回答模板，需要应用代码读取后放进消息，例如把对比表放进系统消息，把模板消息作为用户消息。

LangChain 官方的迁移指南也说明：`langchain.mcp` 目前只封装 Tools，还没有封装 Prompts 和 Resources；需要时直接使用 FastMCP Client 的 `read_resource()` 和 `get_prompt()`。

> ⚠️ **与旧版教程的差异**：LangChain 1.4.0 之前使用 `langchain-mcp-adapters` 包，它提供 `get_resources()`、`load_mcp_resources()`（把资料转成 LangChain 的 Blob）和 `get_prompt()`、`load_mcp_prompt()`（把模板转成 LangChain 消息）。当前的 `langchain.mcp` 没有这些方法。遇到使用这些接口的教程时，改用本节的 FastMCP Client 写法。

5.8 将补齐“客户端获取 → 组装 Agent 输入 → 观察实际消息”的过程，并用本地读取工具验证按需获取。

## 5.7 整理为服务脚本

项目中已保存 [servers/plan_server.py](servers/plan_server.py)。它保留同样的服务名和三类能力，把 `call_log` 换成 stderr 日志，并沿用计算服务的 `--transport stdio/http` 启动入口。HTTP 默认端口是 8001，与计算服务的 8000 错开，方便两个服务同时运行。

与第三节 3.6.2 一样，用 `runpy.run_path` 取得脚本中的 `mcp`，核对它与 notebook 中的定义一致。

In [11]:
script_server = runpy.run_path(str(SCRIPT_FILE))["mcp"]

async with Client(script_server) as script_client:
    script_tools = await script_client.list_tools()
    script_resources = await script_client.list_resources()
    script_prompts = await script_client.list_prompts()
    script_catalog = await script_client.read_resource("plans://catalog")
    script_prompt = await script_client.get_prompt("plan_answer", {"question": question})

checks = {
    "服务名称相同": script_server.name == plan_server.name,
    "工具 Schema 相同":
        {item.name: item.input_schema for item in script_tools}
        == {item.name: item.input_schema for item in tool_specs},
    "资料 URI 相同":
        [str(item.uri) for item in script_resources]
        == [str(item.uri) for item in resource_specs],
    "对比表内容相同": script_catalog[0].text == first_content.text,
    "模板参数相同":
        [item.model_dump() for item in script_prompts[0].arguments]
        == [item.model_dump() for item in prompt_specs[0].arguments],
    "模板消息相同":
        script_prompt.messages[0].content.text == first_message.content.text,
}
for label, passed in checks.items():
    print(f"{label}: {passed}")
assert all(checks.values()), checks

服务名称相同: True
工具 Schema 相同: True
资料 URI 相同: True
对比表内容相同: True
模板参数相同: True
模板消息相同: True


如需手动启动，在**项目根目录的终端**执行（参数含义见第四节 4.3）：

```bash
uv run python Chapter11_MCP/servers/plan_server.py --transport http --port 8001
```

## 5.8 把Resource和Prompt交给Agent

前面的“发现”和“获取”只让内容回到 Python 客户端。**模型知道资料存在、模型拿到资料正文、模型能够发起读取，是三个不同步骤**。

| 应用交给模型的内容 | 模型获得什么 |
|---|---|
| Resource 名称、URI 和描述 | 知道有这份资料，尚未获得正文 |
| Resource 正文与展开后的 Prompt 消息 | 已经可以依据资料和回答要求生成答案 |
| 资源目录 + 一个读取工具的定义 | 可以生成 ToolCall，按需读取正文，再根据 ToolMessage 回答 |

本小节用两组真实 Agent 实验把流程补齐：A 由应用提前准备消息；B 把 Resource 封装成本地工具，由模型在需要时发起读取。它们复用同一个套餐服务。

### 5.8.1 准备真实模型与消息观察函数

复用第一、二节的 `ChatDeepSeek`、`deepseek-v4-flash` 和项目 `conf/.env`。本小节会请求真实模型：A 通常一次模型请求；B 包含一次问候，以及“提出工具调用 → 依据工具结果回答”两步。模型生成的措辞和调用 ID 每次可能不同。

| 接口 / 对象 | 本实验的输入和返回值 |
|---|---|
| `SystemMessage(content=...)` | 应用设置的回答规则，返回系统消息对象 |
| `HumanMessage(content=...)` | 用户问题或参考资料，返回用户消息对象 |
| `AIMessage(content=...)` | 转换 MCP 模板中的 assistant 文本消息；实际模型回复也使用此类 |
| `create_agent(model=..., tools=...)` | 构造 Agent；本身不请求模型 |
| `await agent.ainvoke({"messages": [...]}, config=...)` | 执行对话，返回状态字典；`result["messages"]` 是实际消息序列 |
| `AIMessage.tool_calls` / `ToolMessage` | 前者记录模型提出的工具名、参数与 ID；后者记录执行内容、状态和对应的 tool_call_id |

观察函数打印消息类型、正文与工具调用，不手工制造模型回复或 ToolCall。

In [12]:
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_deepseek import ChatDeepSeek
from mcp.types import GetPromptResult, TextResourceContents

ENV_FILE = PROJECT_ROOT / "conf" / ".env"
load_dotenv(dotenv_path=ENV_FILE, override=True)
if not os.getenv("DEEPSEEK_API_KEY"):
    raise ValueError(f"请在 {ENV_FILE} 中配置 DEEPSEEK_API_KEY，再运行 5.8。")

demo_model = ChatDeepSeek(
    model="deepseek-v4-flash",
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},
    timeout=45,
    max_retries=1,
)


def show_agent_messages(messages: list[BaseMessage]) -> None:
    for index, message in enumerate(messages):
        print(f"[{index}] {type(message).__name__}")
        if isinstance(message, AIMessage) and message.tool_calls:
            print("tool_calls:", json.dumps(message.tool_calls, ensure_ascii=False))
        if isinstance(message, ToolMessage):
            print("工具:", message.name, "状态:", message.status)
            print("tool_call_id:", message.tool_call_id)
        print(message.text)
        print()


def resource_text(contents: list[TextResourceContents]) -> str:
    if not all(isinstance(item, TextResourceContents) for item in contents):
        raise TypeError("本实验仅处理文本 Resource。")
    return "\n\n".join(item.text for item in contents)


def prompt_text_messages(result: GetPromptResult) -> list[BaseMessage]:
    message_classes = {"user": HumanMessage, "assistant": AIMessage}
    converted = []
    for message in result.messages:
        if message.content.type != "text":
            raise TypeError("本实验仅转换文本 Prompt 消息。")
        converted.append(message_classes[message.role](content=message.content.text))
    return converted


print("真实模型:", demo_model.model_name)

真实模型: deepseek-v4-flash


### 5.8.2 方式A：应用获取资料和模板，组装messages

![Resource 和 Prompt 如何组成 Agent 输入消息](assets/20-resource-prompt-agent-messages.svg)

应用先读取 `plans://catalog`，再取得填好问题的 `plan_answer`。本节 Resource 返回文本内容列表，Prompt 返回 MCP 的 PromptMessage；它们都需要明确转换为 LangChain 消息。

- 套餐表作为一条 HumanMessage 参考资料，并附上来源 URI。
- 模板中的 user 转为 HumanMessage，assistant 转为 AIMessage，保留原角色。本例只有 user 消息。
- 应用另外创建 SystemMessage，设置回答规则。本实验的转换函数只处理文本，图片或嵌入资料需要按内容类型另行处理。

本组 Agent 使用已准备的资料，工具列表设为空。模型收到的是消息正文，直接依据对比表回答人数问题。

In [13]:
preloaded_start = len(call_log)
preloaded_question = "专业版最多能加多少人？"

async with Client(plan_server) as preparation_client:
    prepared_resource = await preparation_client.read_resource("plans://catalog")
    prepared_prompt = await preparation_client.get_prompt(
        "plan_answer", {"question": preloaded_question, "style": "简洁"}
    )

prepared_catalog_text = resource_text(prepared_resource)
converted_prompt_messages = prompt_text_messages(prepared_prompt)
prepared_messages = [
    SystemMessage(content="请依据提供的套餐参考资料用中文简洁回答，人数使用阿拉伯数字。"),
    HumanMessage(content=f"套餐参考资料，来源 plans://catalog：\n{prepared_catalog_text}"),
    *converted_prompt_messages,
]

preparation_records = call_log[preloaded_start:]
print("Prompt 原角色与转换后的类型:", [
    (original.role, type(converted).__name__)
    for original, converted in zip(prepared_prompt.messages, converted_prompt_messages)
])
print("准备的输入消息类型:", [type(message).__name__ for message in prepared_messages])
print("模型调用前，服务端已执行:", preparation_records)

Prompt 原角色与转换后的类型: [('user', 'HumanMessage')]
准备的输入消息类型: ['SystemMessage', 'HumanMessage', 'HumanMessage']
模型调用前，服务端已执行: [('resource', 'plans://catalog'), ('prompt', 'plan_answer', '简洁')]


In [14]:
preloaded_agent = create_agent(model=demo_model, tools=[])
preloaded_model_start = len(call_log)
preloaded_response = await preloaded_agent.ainvoke(
    {"messages": prepared_messages}, config={"recursion_limit": 8}
)

print("方式 A 的实际消息:")
show_agent_messages(preloaded_response["messages"])
preloaded_tool_calls = [
    call for message in preloaded_response["messages"]
    if isinstance(message, AIMessage) for call in message.tool_calls
]
preloaded_model_records = call_log[preloaded_model_start:]
print("模型生成的工具请求次数:", len(preloaded_tool_calls))
print("模型运行期间新增的 MCP 服务调用:", preloaded_model_records)

方式 A 的实际消息:
[0] SystemMessage
请依据提供的套餐参考资料用中文简洁回答，人数使用阿拉伯数字。

[1] HumanMessage
套餐参考资料，来源 plans://catalog：
| 套餐 | 价格 | 成员人数上限 | 月度AI问答额度 | API调用额度 |
|---|---|---|---|---|
| 试用版 | 0 元，注册后 14 天 | 1 人 | 200 次 | 不支持 |
| 基础版 | 99 元 / 用户 / 月 | 10 人 | 5000 次 | 每月 10000 次 |
| 专业版 | 199 元 / 用户 / 月 | 50 人 | 30000 次 | 每月 80000 次 |
| 企业版 | 按年签约，不公开标价 | 按合同约定 | 按合同约定 | 按合同约定 |

[2] HumanMessage
你是 atguigu助手的客服。请只依据套餐资料回答用户的问题；资料没有写明的内容，说明需要人工确认。
回答风格：简洁。
用户问题：专业版最多能加多少人？

[3] AIMessage
专业版成员人数上限为50人。

模型生成的工具请求次数: 0
模型运行期间新增的 MCP 服务调用: []


实际消息中可以看到：套餐表是 HumanMessage，客服模板也是 HumanMessage，最后追加真实 AIMessage。模型已经收到“专业版 50 人”的资料，直接回答；Resource 和 Prompt 的服务函数在模型请求前由应用执行，本组不会产生读取 ToolMessage。

### 5.8.3 方式B：把Resource读取封装成LangChain工具

![模型按需读取 MCP Resource 的消息和调用过程](assets/21-resource-as-demand-tool.svg)

新建一个应用侧的异步工具 `read_plan_catalog`，其内部通过 Client 调用 `read_resource`。这里使用第五章的 LangChain `@tool`；服务端仍使用 `@plan_server.resource` 注册套餐表。

| 项目 | 本例含义 |
|---|---|
| 工具输入 `uri` | 用 Literal 限定为本例的 `plans://catalog`，模型按 Schema 填写 |
| 工具实现 | 建立 Client 会话 → 读取 Resource → 合并文本 |
| 函数返回 `str` | 资料正文与来源；Agent 工具执行环节将其包装为 ToolMessage |
| 读取时机 | 模型生成 ToolCall 之后；定义工具、查看 Schema 不读取正文 |

`MCPAdapter` 不会自动创建这个包装工具，需要应用明确编写并加入 Agent。查看服务端 Tool 名单与模型会收到的包装工具定义，确认两层各自提供了什么。

In [ ]:
# 应用层封装到工具函数，然后绑定给模型
resource_tool_reads = []


@tool
async def read_plan_catalog(uri: Literal["plans://catalog"]) -> str:
    """读取套餐对比表，用于查询或比较套餐价格、成员人数上限和调用额度。"""
    resource_tool_reads.append(uri)
    async with Client(plan_server) as resource_client:
        contents = await resource_client.read_resource(uri)
    return f"资料来源：{uri}\n{resource_text(contents)}"


async with Client(plan_server) as inspection_client:
    server_tool_specs = await inspection_client.list_tools()

print("MCP 服务端的 Tool 名单:", [item.name for item in server_tool_specs])
print("应用侧读取工具:", type(read_plan_catalog).__name__, read_plan_catalog.name)
print("模型收到的工具定义:")
print(json.dumps(convert_to_openai_tool(read_plan_catalog), ensure_ascii=False, indent=2))
print("定义与检查后，包装工具的读取记录:", resource_tool_reads)

MCP 服务端的 Tool 名单: ['get_plan']
应用侧读取工具: StructuredTool read_plan_catalog
模型收到的工具定义:
{
  "type": "function",
  "function": {
    "name": "read_plan_catalog",
    "description": "读取套餐对比表，用于查询或比较套餐价格、成员人数上限和调用额度。",
    "parameters": {
      "properties": {
        "uri": {
          "const": "plans://catalog",
          "type": "string"
        }
      },
      "required": [
        "uri"
      ],
      "type": "object"
    }
  }
}
定义与检查后，包装工具的读取记录: []


### 5.8.4 先给资源目录，正文等调用后再提供

把 5.4 发现的资源名称、URI 和描述放入 SystemMessage，让模型知道有哪些资料可读。目录只有元数据；完整套餐表等读取工具返回后才进入对话。

本组 Agent 绑定 `read_plan_catalog`。规则要求：当前对话缺少套餐资料时先读取；已有正文时使用现有资料；日常问候直接回复。两个案例分别从新的输入消息开始，便于独立核对调用记录。

In [16]:
resource_directory = [
    {"uri": str(item.uri), "name": item.name, "description": item.description}
    for item in resource_specs
]
demand_system_message = SystemMessage(content=(
    "你是套餐咨询助手，请用中文简洁回答，数字使用阿拉伯数字。\n"
    "查询或比较套餐事实时，只能依据当前对话中的套餐资料。"
    "如果当前对话没有资料正文，先用 read_plan_catalog 读取相关资料。"
    "资料已在对话中时可以直接使用。资料未说明的内容需要人工确认。"
    "日常问候直接回复，无需读取套餐资料。\n"
    "可用资源目录（这里是描述，不是正文）：\n"
    + json.dumps(resource_directory, ensure_ascii=False, indent=2)
))
demand_agent = create_agent(model=demo_model, tools=[read_plan_catalog])

print("模型开始时看到的资源目录:")
print(json.dumps(resource_directory, ensure_ascii=False, indent=2))
print("绑定到 Agent 的工具:", read_plan_catalog.name)
print("Agent 创建后，包装工具读取次数:", len(resource_tool_reads))

模型开始时看到的资源目录:
[
  {
    "uri": "plans://catalog",
    "name": "plan_catalog",
    "description": "四档标准订阅套餐的对比表。"
  }
]
绑定到 Agent 的工具: read_plan_catalog
Agent 创建后，包装工具读取次数: 0


**小实验一：日常问候。** 先预测是否需要读取套餐表，再查看真实模型回复和服务端记录。

In [17]:
greeting_start = len(call_log)
greeting_read_start = len(resource_tool_reads)
greeting_response = await demand_agent.ainvoke(
    {"messages": [demand_system_message, HumanMessage(content="你好，请用一句话打个招呼。") ]},
    config={"recursion_limit": 8},
)
greeting_records = call_log[greeting_start:]
greeting_reads = resource_tool_reads[greeting_read_start:]

show_agent_messages(greeting_response["messages"])
print("本次包装工具的读取记录:", greeting_reads)
print("本次 MCP 服务端调用记录:", greeting_records)

[0] SystemMessage
你是套餐咨询助手，请用中文简洁回答，数字使用阿拉伯数字。
查询或比较套餐事实时，只能依据当前对话中的套餐资料。如果当前对话没有资料正文，先用 read_plan_catalog 读取相关资料。资料已在对话中时可以直接使用。资料未说明的内容需要人工确认。日常问候直接回复，无需读取套餐资料。
可用资源目录（这里是描述，不是正文）：
[
  {
    "uri": "plans://catalog",
    "name": "plan_catalog",
    "description": "四档标准订阅套餐的对比表。"
  }
]

[1] HumanMessage
你好，请用一句话打个招呼。

[2] AIMessage
你好！很高兴为你服务，有什么套餐问题可以随时问我。

本次包装工具的读取记录: []
本次 MCP 服务端调用记录: []


**小实验二：换成套餐对比。** 问题需要具体人数和额度；输入仍只有资源目录及问题。观察模型是否生成带 URI 的 ToolCall，工具返回的正文如何成为 ToolMessage，以及模型如何据此回答。

In [18]:
comparison_start = len(call_log)
comparison_read_start = len(resource_tool_reads)
comparison_response = await demand_agent.ainvoke(
    {"messages": [
        demand_system_message,
        HumanMessage(content="请比较基础版和专业版的成员人数上限及 API 调用额度。"),
    ]},
    config={"recursion_limit": 8},
)
comparison_records = call_log[comparison_start:]
comparison_reads = resource_tool_reads[comparison_read_start:]

show_agent_messages(comparison_response["messages"])
print("本次包装工具的读取记录:", comparison_reads)
print("本次 MCP 服务端调用记录:", comparison_records)

[0] SystemMessage
你是套餐咨询助手，请用中文简洁回答，数字使用阿拉伯数字。
查询或比较套餐事实时，只能依据当前对话中的套餐资料。如果当前对话没有资料正文，先用 read_plan_catalog 读取相关资料。资料已在对话中时可以直接使用。资料未说明的内容需要人工确认。日常问候直接回复，无需读取套餐资料。
可用资源目录（这里是描述，不是正文）：
[
  {
    "uri": "plans://catalog",
    "name": "plan_catalog",
    "description": "四档标准订阅套餐的对比表。"
  }
]

[1] HumanMessage
请比较基础版和专业版的成员人数上限及 API 调用额度。

[2] AIMessage
tool_calls: [{"name": "read_plan_catalog", "args": {"uri": "plans://catalog"}, "id": "call_00_baHkC4TzUlZ9wvcp9g1p2533", "type": "tool_call"}]


[3] ToolMessage
工具: read_plan_catalog 状态: success
tool_call_id: call_00_baHkC4TzUlZ9wvcp9g1p2533
资料来源：plans://catalog
| 套餐 | 价格 | 成员人数上限 | 月度AI问答额度 | API调用额度 |
|---|---|---|---|---|
| 试用版 | 0 元，注册后 14 天 | 1 人 | 200 次 | 不支持 |
| 基础版 | 99 元 / 用户 / 月 | 10 人 | 5000 次 | 每月 10000 次 |
| 专业版 | 199 元 / 用户 / 月 | 50 人 | 30000 次 | 每月 80000 次 |
| 企业版 | 按年签约，不公开标价 | 按合同约定 | 按合同约定 | 按合同约定 |

[4] AIMessage
| 项目 | 基础版 | 专业版 |
|---|---|---|---|
| 价格 | 99 元 / 用户 / 月 | 199 元 / 用户 / 月 |
| 成员人数上限 | 10 人 | 50 人 |
| 月度AI问答额度

### 5.8.5 对齐输入、ToolCall、ToolMessage和服务端记录

不能只看最后一句回答。核对输入包含什么、模型提出了哪些请求、调用 ID 是否对应、服务端是否实际读取 Resource。

A 的读取发生在模型请求前；B 的问候可以没有读取，套餐对比则需要先收到资料正文。下面的检查用于验证这两条实际路径；模型回答的措辞不要求逐字相同。

In [19]:
greeting_calls = [
    call for message in greeting_response["messages"]
    if isinstance(message, AIMessage) for call in message.tool_calls
]
comparison_calls = [
    call for message in comparison_response["messages"]
    if isinstance(message, AIMessage) for call in message.tool_calls
]
comparison_tool_messages = [
    message for message in comparison_response["messages"] if isinstance(message, ToolMessage)
]
requested_ids = {call["id"] for call in comparison_calls}
returned_ids = {message.tool_call_id for message in comparison_tool_messages}

checks = {
    "A 输入消息包含 Resource 正文": prepared_catalog_text in prepared_messages[1].text,
    "A 输入消息包含展开后的 Prompt":
        prepared_prompt.messages[0].content.text == converted_prompt_messages[0].text,
    "A 由应用先读取 Resource 和 Prompt":
        [entry[0] for entry in preparation_records] == ["resource", "prompt"],
    "A 模型直接回答 50 人": "50" in preloaded_response["messages"][-1].text,
    "A 模型阶段没有工具请求和 MCP 读取": not preloaded_tool_calls and not preloaded_model_records,
    "B 问候没有读取资料": not greeting_calls and not greeting_reads and not greeting_records,
    "B 套餐问题生成真实读取 ToolCall": bool(comparison_calls) and all(
        call["name"] == "read_plan_catalog" and call["args"] == {"uri": "plans://catalog"}
        for call in comparison_calls
    ),
    "B 请求 ID 与工具消息 ID 对应": requested_ids == returned_ids,
    "B 工具消息成功且包含 Resource 正文": bool(comparison_tool_messages) and all(
        message.status == "success" and prepared_catalog_text in message.text
        for message in comparison_tool_messages
    ),
    "B 包装器与 MCP 实际读取次数对应":
        len(comparison_reads) == len(comparison_records) == len(comparison_calls)
        and all(entry[0] == "resource" for entry in comparison_records),
    "B 模型得到资料后生成回答": bool(comparison_response["messages"][-1].text)
        and "50" in comparison_response["messages"][-1].text,
}
for label, passed in checks.items():
    print(f"{label}: {passed}")
assert all(checks.values()), checks

A 输入消息包含 Resource 正文: True
A 输入消息包含展开后的 Prompt: True
A 由应用先读取 Resource 和 Prompt: True
A 模型直接回答 50 人: True
A 模型阶段没有工具请求和 MCP 读取: True
B 问候没有读取资料: True
B 套餐问题生成真实读取 ToolCall: True
B 请求 ID 与工具消息 ID 对应: True
B 工具消息成功且包含 Resource 正文: True
B 包装器与 MCP 实际读取次数对应: True
B 模型得到资料后生成回答: True


| 比较项 | A：应用提前准备 | B：模型按需读取 |
|---|---|---|
| 模型请求前拿到什么 | 资料正文、展开后的 Prompt、应用规则 | 资源目录、读取工具定义、问题与规则 |
| 谁决定读取时机 | 应用在请求模型前读取 | 模型生成 ToolCall，Agent 执行包装工具 |
| 资料以什么消息进入 | HumanMessage 参考资料 | 工具返回的 ToolMessage |
| Prompt 怎样使用 | 按原角色转为 LangChain 消息 | 本组由应用直接设置规则；需要模板时仍由应用获取并转换 |
| 是否改变 MCP 服务端 Resource 注册 | 不需要 | 不需要，包装工具定义在应用侧 |

“不自动转换”说明 MCPAdapter 的封装范围。应用仍可以把 Resource、Prompt 组成消息；也可以开放一个读取工具，让模型提出获取正文的请求。

小练习：把 A 的 `style` 改为“分点列出”，查看转换后的消息与真实回答；再把 B 的问题改成另一个套餐问题，检查 URI、ToolCall、ToolMessage 和读取记录。不要只根据最终回答猜测工具是否执行。

参考：[LangChain 消息](https://docs.langchain.com/oss/python/langchain/messages)、[LangChain 工具](https://docs.langchain.com/oss/python/langchain/tools)、[FastMCP 读取资源](https://gofastmcp.com/clients/resources)、[FastMCP 获取模板](https://gofastmcp.com/clients/prompts)。

## 5.9 小结与小练习

| 能力 | 由谁决定使用 | 服务端注册 | 客户端发现 | 客户端获取 | MCPAdapter |
|---|---|---|---|---|---|
| Tools | 模型 | `@mcp.tool` | `list_tools()` | `call_tool()` → `CallToolResult` | 转成 `StructuredTool`，进入 Agent 的工具列表 |
| Resources | 应用 | `@mcp.resource(uri)` | `list_resources()` | `read_resource()` → 内容列表 | 不自动转换；由应用组成消息或封装读取工具 |
| Prompts | 用户 | `@mcp.prompt` | `list_prompts()` | `get_prompt()` → `GetPromptResult` | 不自动转换；获取后由应用转换为消息 |

小练习：

1. 把 5.5.2 的 `style` 改成“分点列出”。预测消息文本与 `call_log` 的变化，再验证。
2. 读取一个不存在的 URI，例如 `plans://refund`，观察客户端抛出的异常。`read_resource` 没有 `raise_on_error` 参数，对照第三节 3.5，说明它与工具错误的处理有什么不同。
3. 用 `@plan_server.resource("plans://{plan}")` 注册资源模板，函数接收 `plan` 参数，返回单个套餐的 Markdown。分别用 `list_resources()` 和 `list_resource_templates()` 查看，再读取 `plans://专业版`。
4. 判断下面三种情况分别适合哪类能力：用户问“专业版能加多少人”；界面上提供“套餐答疑”按钮；应用在每次对话前附上套餐对比表。

下一节同时接入计算服务和本节的套餐服务，混合本地工具，并观察连接失败。

参考：[MCP 规范：服务端能力概览](https://modelcontextprotocol.io/specification/2025-11-25/server)、[FastMCP Resources](https://gofastmcp.com/servers/resources)、[FastMCP Prompts](https://gofastmcp.com/servers/prompts)、[客户端读取资料](https://gofastmcp.com/clients/resources)、[客户端获取模板](https://gofastmcp.com/clients/prompts)、[LangChain MCP 迁移指南](https://docs.langchain.com/oss/python/migrate/langchain-mcp-adapters)。接口与输出以本节打印的环境版本为准。